In [1]:
%%capture
!pip install open-clip-torch timm transformers ftfy regex -q

In [2]:
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import time
from tqdm.auto import tqdm
import gc
import open_clip
import os
import pickle

In [3]:
class Embedder:
    """
    CLIP Embedder được đơn giản hóa.
    Chỉ có nhiệm vụ duy nhất là encode một batch ảnh được đưa cho.
    """
    
    def __init__(self, device, model_name, pretrained, tokenizer_model, use_multi_gpu=False):
        self.device = device
        self.model_name = model_name
        self.pretrained = pretrained
        self.tokenizer_model = tokenizer_model
        self.use_multi_gpu = use_multi_gpu
        self._load_model()
        self._setup_multi_gpu()
        
    def _load_model(self):
        """
        Tải mô hình và quan trọng nhất là chuyển nó đến đúng device.
        """
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            self.model_name, pretrained=self.pretrained)
        self.model = self.model.to(self.device)
        self.model.eval()
    
    def _setup_multi_gpu(self):
        """
        Nếu có nhiều GPU, bọc mô hình trong DataParallel.
        """
        if self.use_multi_gpu and torch.cuda.device_count() > 1:
            print(f"Multi-GPU enabled: {torch.cuda.device_count()} GPUs")
            self.model = torch.nn.DataParallel(self.model)
    
    def _get_actual_model(self):
        return self.model.module if hasattr(self.model, 'module') else self.model
    
    def encode_images_batch(self, images_batch):
        """
        Encode một batch ảnh PIL. Trả về mảng numpy của các embedding.
        """
        if not images_batch:
            return None
        
        try:
            image_tensors = torch.stack([self.preprocess(img) for img in images_batch]).to(self.device)
            
            with torch.amp.autocast(self.device.type):
                with torch.no_grad():
                    batch_embeddings = self._get_actual_model().encode_image(image_tensors)
                    batch_embeddings_norm = torch.nn.functional.normalize(batch_embeddings, p=2, dim=-1)
            
            results = batch_embeddings_norm.cpu().numpy()
            
            del image_tensors, batch_embeddings, batch_embeddings_norm
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
            
            return results
            
        except Exception as e:
            print(f"Batch encoding failed: {e}")
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
            return None

    def cleanup(self):
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()

In [4]:
class EmbeddingProcessor:
    """
    Xử lý một tập dữ liệu gồm các khung hình của video để tạo và lưu các embedding.
    """
    
    def __init__(self, embedder, output_dir, batch_size, progress_config=None):
        self.embedder = embedder
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.batch_size = batch_size
        self.progress_config = progress_config or {}
        self.embedding_info = {
            'paths': [],
            'embeddings': [],
            'length': 0
        }

        print(f"📁 Output directory: {self.output_dir}")
        print(f"📦 Batch size: {self.batch_size}")
        if self.progress_config:
            start = self.progress_config.get('start', 'N/A')
            end = self.progress_config.get('end', 'N/A')
            print(f"Progress control: Start={start}, End={end}")
        else:
            print("No progress control. Will process all videos.")

    def process_dataset(self, dataset_path):
        """
        Quét dataset, tạo embedding cho tất cả các khung hình theo batch,
        và lưu trữ chúng trong dictionary self.embedding_info.
        """
        all_video_folders = sorted([p for p in Path(dataset_path).iterdir() if p.is_dir()])
        
        start_index = self.progress_config.get('start', 0)
        end_index = self.progress_config.get('end', len(all_video_folders))
        videos_to_process = all_video_folders[start_index:end_index]
        
        if not videos_to_process:
            print("No videos to process in the specified range.")
            return

        print(f"Processing {len(videos_to_process)} videos from index {start_index} to {end_index-1}.")
        
        image_buffer = []
        metadata_buffer = []

        for video_folder in tqdm(videos_to_process, desc="Aggregating and Embedding"):
            video_name = video_folder.name
            frame_paths = sorted(video_folder.glob("*.jpg"), key=lambda x: int(x.stem))
            
            for path in frame_paths:
                image_buffer.append(Image.open(path).convert("RGB"))
                # Tạo đường dẫn tương đối video_name/frame_name.jpg
                relative_path = f"{video_name}/{path.name}"
                metadata_buffer.append({'path': relative_path})

                if len(image_buffer) >= self.batch_size:
                    batch_embeddings = self.embedder.encode_images_batch(image_buffer)
                    if batch_embeddings is not None:
                        self.embedding_info['paths'].extend([meta['path'] for meta in metadata_buffer])
                        self.embedding_info['embeddings'].extend(list(batch_embeddings))
                    
                    image_buffer.clear()
                    metadata_buffer.clear()

        # Xử lý batch cuối cùng còn sót lại
        if image_buffer:
            print(f"Flushing final batch of {len(image_buffer)} images...")
            batch_embeddings = self.embedder.encode_images_batch(image_buffer)
            if batch_embeddings is not None:
                self.embedding_info['paths'].extend([meta['path'] for meta in metadata_buffer])
                self.embedding_info['embeddings'].extend(list(batch_embeddings))

        print("Embedding process completed.")
        self.embedder.cleanup()

    def save_embeddings(self):
        """
        Lưu dictionary embedding_info đã được điền dữ liệu vào một file pickle.
        """
        if not self.embedding_info['paths']:
            print("⚠️ No embeddings were generated. Nothing to save.")
            return

        # Cập nhật số lượng và chuyển đổi list embedding thành một mảng NumPy
        self.embedding_info['length'] = len(self.embedding_info['paths'])
        self.embedding_info['embeddings'] = np.array(self.embedding_info['embeddings'], dtype=np.float32)
        
        output_file = self.output_dir / "embedding_info.pkl"
        
        print(f"💾 Saving {self.embedding_info['length']} embeddings to {output_file}...")
        
        try:
            with open(output_file, 'wb') as f:
                pickle.dump(self.embedding_info, f)
            print(f"Successfully saved embedding_info.pkl.")
        except Exception as e:
            print(f"Failed to save embeddings: {e}")

In [5]:
class EmbeddingProcessor:
    """
    Quét một tập dữ liệu ảnh, tạo embedding, và lưu kết quả.
    Việc quét, sắp xếp và chuẩn bị danh sách ảnh được thực hiện ngay khi khởi tạo.
    """
    
    def __init__(self, embedder, input_path, output_dir, batch_size, progress_config=None):
        self.embedder = embedder
        self.input_path = Path(input_path)
        self.output_dir = Path(output_dir)
        self.batch_size = batch_size
        self.progress_config = progress_config or {}
        
        self.output_dir.mkdir(exist_ok=True, parents=True)

        if not self.input_path.is_dir():
            raise ValueError(f"Đường dẫn đầu vào không tồn tại hoặc không phải là thư mục: {self.input_path}")

        # Quét và sắp xếp
        all_image_paths = []
        for folder_path in self.input_path.iterdir():
            if folder_path.is_dir():
                all_image_paths.extend(folder_path.glob('*.jpg'))
        all_image_paths.sort(key=lambda p: (p.parent.name, int(p.stem)))
        
        self.progress_config["length"] = len(all_image_paths)

        # Áp dụng progress để xác định danh sách cần xử lý
        start_index = self.progress_config.get('start', 0)
        end_index = self.progress_config.get('end', self.progress_config["length"])
        self.progress_config["start"] = start_index
        self.progress_config["end"] = end_index

        print(f"progress: {self.progress_config}")
        self.paths_to_process = all_image_paths[start_index:end_index]

        # Chuẩn bị dictionary để lưu kết quả
        self.embedding_info = {
            'paths': [],
            'embeddings': [],
            'length': 0
        }

    def _get_relative_path(self, full_path):
        """Trích xuất phần 'video_name/frame_index.jpg' từ đường dẫn đầy đủ."""
        p = Path(full_path)
        return f"{p.parent.name}/{p.name}"

    def process_dataset(self):
        """
        Thực hiện quá trình embedding trên danh sách ảnh đã được chuẩn bị và lưu kết quả.
        """
        if not self.paths_to_process:
            print("Không có ảnh nào để xử lý. Dừng quá trình.")
            return

        print("\n--- STAGE: Embedding images ---")
        num_to_process = len(self.paths_to_process)
        
        for i in tqdm(range(0, num_to_process, self.batch_size), desc="Embedding Batches"):
            batch_paths = self.paths_to_process[i:i + self.batch_size]
            
            try:
                batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
            except Exception as e:
                print(f"Lỗi khi đọc ảnh trong batch, bỏ qua batch này: {e}")
                continue

            batch_embeddings = self.embedder.encode_images_batch(batch_images)
            
            if batch_embeddings is not None:
                relative_paths = [self._get_relative_path(p) for p in batch_paths]
                self.embedding_info['paths'].extend(relative_paths)
                self.embedding_info['embeddings'].extend(list(batch_embeddings))

            del batch_images, batch_embeddings
            gc.collect()

        print("✅ Embedding process completed.")
        self.embedder.cleanup()
        
        # Tự động gọi hàm lưu sau khi embedding xong
        self._save_embeddings()

    def _save_embeddings(self):
        """
        [Hàm nội bộ] Lưu dictionary embedding_info vào file pickle.
        """
        if not self.embedding_info['paths']:
            print("⚠️ Không có embedding nào được tạo ra. Không có gì để lưu.")
            return

        self.embedding_info['length'] = len(self.embedding_info['paths'])
        self.embedding_info['embeddings'] = np.array(self.embedding_info['embeddings'], dtype=np.float32)
        
        output_file = self.output_dir / "embedding_info.pkl"
        
        print(f"\n--- STAGE: Saving results ---")
        print(f"💾 Saving {self.embedding_info['length']} embeddings to {output_file}...")
        
        try:
            with open(output_file, 'wb') as f:
                pickle.dump(self.embedding_info, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"✅ Successfully saved {output_file}.")
        except Exception as e:
            print(f"❌ Failed to save embeddings: {e}")

In [6]:
try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_count = torch.cuda.device_count()
        
        print(f"GPU Count: {gpu_count}")
        
        # Multi-GPU strategy
        if gpu_count > 1:
            print(f"Multi-GPU detected! {gpu_count} GPUs available")
            use_multi_gpu = True
        else:
            print(f"Single GPU: {torch.cuda.get_device_name(0)}")
            use_multi_gpu = False
        
        # Clear GPU cache
        torch.cuda.empty_cache()
        
    else:
        device = torch.device("cpu")
        gpu_count = 0
        use_multi_gpu = False
        print("GPU not available!")        
except Exception as e:
    print(f"GPU setup failed: {e}")
    device = torch.device("cpu")
    gpu_count = 0
    use_multi_gpu = False

print(f"Using device: {device}")
print("GPU setup completed!")

GPU Count: 1
Single GPU: Tesla P100-PCIE-16GB
Using device: cuda
GPU setup completed!


In [ ]:
# --- Cấu hình đường dẫn ---
# !!! QUAN TRỌNG: Thay đổi đường dẫn này để trỏ đến thư mục chứa các thư mục video/frame của bạn
DATASET_PATH = "/kaggle/input/kf-full" 
# Thư mục để lưu file embedding_info.pkl
OUTPUT_PATH = "/kaggle/working/aic-2025-embeddings" 

# --- Cấu hình mô hình & Batch ---
EMBEDDER_MODEL_NAME = "ViT-H-14-378-quickgelu"
EMBEDDER_PRETRAINED = "dfn5b"
EMBEDDER_TOKENIZER_MODEL = "ViT-H-14-378-quickgelu"
BATCH_SIZE = 256

# --- Khởi tạo Embedder ---
try:
    embedder = Embedder(
        device=device, 
        model_name=EMBEDDER_MODEL_NAME,
        pretrained=EMBEDDER_PRETRAINED,
        tokenizer_model=EMBEDDER_TOKENIZER_MODEL,
        use_multi_gpu=use_multi_gpu
    )
    print(f"Embedder ({EMBEDDER_MODEL_NAME}) initialized successfully")
except Exception as e:
    print(f"Failed to initialize embedder: {e}")
    raise e

# --- Cấu hình tiến trình (TÙY CHỌN) ---
# TÙY CHỌN 1: Xử lý một phần của dataset (ví dụ: xử lý 10 thư mục đầu tiên, từ index 0 đến 9)
progress_config = {
    'length': 374251,
    'start': 0,
    'end': 10000 
}

# TÙY CHỌN 2: Xử lý toàn bộ dataset
# progress_config = {}


# --- Khởi tạo và chạy Processor ---
processor = EmbeddingProcessor(
    embedder=embedder,
    output_dir=OUTPUT_PATH,
    input_path=DATASET_PATH,
    batch_size=BATCH_SIZE,
    progress_config=progress_config
)

# Bắt đầu quá trình embedding
processor.process_dataset()

open_clip_pytorch_model.bin:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Embedder (ViT-H-14-378-quickgelu) initialized successfully
progress: {'length': 374251, 'start': 0, 'end': 374251}

--- STAGE: Embedding images ---


Embedding Batches:   0%|          | 0/1462 [00:00<?, ?it/s]